In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.fully_connected import train_model
from src.helper import make_run_dir, save_run

## Load data

In [2]:
DATA_SET = 'rand_C'

df_train = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_train.parquet'))
df_val   = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_val.parquet'))
df_test  = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_test.parquet'))


OUTPUT_DIR = make_run_dir()

## Hyperparameters & Features

In [3]:
BATCH_SIZE    = 4096
MAX_EPOCHS    = 100
PATIENCE      = 30
LR_PATIENCE   = 8
LR_FACTOR     = 0.3
INIT_LR       = 1e-3
ACTIVATION    = 'relu'
NEURONS        = 80
HIDDEN_LAYERS  = 3


FEATURES_3 = ['delta', 'T', 'spy_ret']
FEATURES_4 = ['delta', 'T', 'spy_ret', 'vix_lag']
TARGET      = 'd_iv'

## Analytic benchmark

In [4]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 64.9134  RMSE = 0.016269
Coefficients: a = -0.078536, b = -0.092386, c = -0.081882


## 3-Feature & 3-Feature ANN 

In [5]:
train_cfg = dict(
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, lr=INIT_LR,
    patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
    hidden_layers=HIDDEN_LAYERS, neurons=NEURONS, activation=ACTIVATION,
    target=TARGET,
)

result_3f = train_model(df_train, df_val, df_test,features=FEATURES_3, desc="ANN 3F", **train_cfg)

result_4f = train_model(df_train, df_val, df_test, features=FEATURES_4, desc="ANN 4F", **train_cfg)

ANN 3F:   0%|          | 0/100 epochs [00:00<?, ?epoch/s]  


Test:
SSE = 57.3609  RMSE = 0.015293  Time = 93.9s


ANN 4F:   0%|          | 0/100 epochs [00:00<?, ?epoch/s]  


Test:
SSE = 43.0808  RMSE = 0.013254  Time = 93.2s


In [6]:
summary = save_run(
    run_dir=OUTPUT_DIR,
    y_test=df_test[TARGET].values,
    hw=hw,
    models={"ANN-3F": result_3f, "ANN-4F": result_4f},
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,64.913437,0.000265,0.016269,0.005912,1.246231e-03,0.002637,0.068266,NaN,NaN,NaN
1,ANN-3F,57.360855,0.000234,0.015293,0.005749,-3.236244e-07,0.002504,0.176672,93.9s,11.63%,NaN
2,ANN-4F,43.080784,0.000176,0.013254,0.005546,-3.830244e-05,0.002626,0.381640,93.2s,33.63%,24.90%
3,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,187.1s,NaN,NaN
